# Multimodal Retrieval-Augmented Generation System

Unified retrieval across text, code, and images with hybrid retrieval, reranking, and evaluation-driven design.

In [ ]:
!pip install -q -r /kaggle/input/p2-kaggle-package/requirements.txt
import sys
sys.path.append('/kaggle/input/p2-kaggle-package')

In [ ]:
from config.settings import Config

Config.DATASET = 'main'

## Data Processing

Chunking splits long documents and code into retrieval-friendly units while preserving section/function context.

In [ ]:
from pathlib import Path
from chunking.text_chunker import chunk_text_file
from chunking.code_chunker import split_code_by_functions
from config.settings import get_path

data_root = Path(get_path('data'))
text_dir = data_root / 'raw' / 'text'
code_dir = data_root / 'raw' / 'code'

for text_file in sorted(text_dir.glob('*.txt')):
    chunk_text_file(text_file)

for code_file in sorted(code_dir.glob('*.py')):
    split_code_by_functions(code_file)

## Embeddings and Indexing

Embeddings map text/code/images into dense vectors. FAISS indexes those vectors for fast nearest-neighbor retrieval.

In [ ]:
!python /kaggle/input/p2-kaggle-package/build_and_test_indexes.py

## Evaluation

Evaluation uses standard retrieval metrics:
- Precision@k
- Recall@k
- NDCG
- MRR

In [ ]:
!python /kaggle/input/p2-kaggle-package/compute_semantic_metrics.py
!python /kaggle/input/p2-kaggle-package/compute_hybrid_metrics.py
!python /kaggle/input/p2-kaggle-package/compute_reranked_metrics.py

In [ ]:
import matplotlib.pyplot as plt

labels = ['Semantic', 'Hybrid', 'Reranked']
mean_p3 = [0.6667, 0.5000, 0.5833]
mean_r5 = [0.9375, 0.7917, 0.8125]
mean_ndcg = [0.9039, 0.7545, 0.7789]

x = range(len(labels))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar([i - 0.25 for i in x], mean_p3, width=0.25, label='P@3')
ax.bar(x, mean_r5, width=0.25, label='R@5')
ax.bar([i + 0.25 for i in x], mean_ndcg, width=0.25, label='NDCG@5')
ax.set_xticks(list(x))
ax.set_xticklabels(labels)
ax.set_ylabel('Score')
ax.set_title('Semantic vs Hybrid vs Reranked')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
query_labels = ['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6', 'Q7', 'Q8']
query_ndcg = [1.0, 1.0, 0.9675, 1.0, 0.2641, 0.0, 1.0, 1.0]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(query_labels, query_ndcg)
ax.set_xlabel('Query')
ax.set_ylabel('NDCG@5')
ax.set_title('Per-query performance (Reranked)')
plt.tight_layout()
plt.show()

## RAG Demo

In [ ]:
from full_rag_system import FullRAGSystem

system = FullRAGSystem()
demo_queries = [
    ('Explain DBSCAN eps parameter', 'text'),
    ('Example of DBSCAN eps tuning in Python', 'code'),
    ('ROC curve visualization example', 'image'),
]

for query, modality in demo_queries:
    result = system.run_query(query, modality)
    print('Query:')
    print(result['query'])
    print('\nAnswer:')
    print(result['answer'])
    print('\nSources:')
    print(', '.join(result['sources']))
    print()